In [1]:
from datetime import date
from pathlib import Path
from pprint import pprint

import pandas as pd

from tradepilot.etl.models import IngestionRequest
from tradepilot.etl.service import ETLService

PROJECT_ROOT = Path("/home/nixos/workspace/TradePilot")
LAKEHOUSE = PROJECT_ROOT / "data/lakehouse"
NORMALIZED_ROOT = LAKEHOUSE / "normalized"
RAW_ROOT = LAKEHOUSE / "raw"

In [2]:
RUN_STAGE_F_SYNC = False

if RUN_STAGE_F_SYNC:
    service = ETLService()
    request = IngestionRequest(
        request_start=date(2026, 4, 20),
        request_end=date(2026, 4, 20),
    )

    daily_result = service.run_dataset_sync("rates.daily_rates", request)
    lpr_result = service.run_dataset_sync("rates.lpr", request)

    pprint(daily_result.model_dump())
    pprint(lpr_result.model_dump())
else:
    daily_result = None
    lpr_result = None
    print("Skipped live Stage F sync; reading existing lakehouse outputs.")

Skipped live Stage F sync; reading existing lakehouse outputs.


In [3]:
def read_partitioned_dataset(root: Path, dataset_name: str) -> tuple[pd.DataFrame, list[Path]]:
    files = sorted((root / dataset_name).glob("*/*/part-00000.parquet"))
    if not files:
        return pd.DataFrame(), files
    frame = pd.concat([pd.read_parquet(path) for path in files], ignore_index=True)
    return frame, files


daily, daily_files = read_partitioned_dataset(NORMALIZED_ROOT, "rates.daily_rates")
lpr, lpr_files = read_partitioned_dataset(NORMALIZED_ROOT, "rates.lpr")

{
    "daily_files": [str(path.relative_to(PROJECT_ROOT)) for path in daily_files],
    "daily_rows": len(daily),
    "lpr_files": [str(path.relative_to(PROJECT_ROOT)) for path in lpr_files],
    "lpr_rows": len(lpr),
}

{'daily_files': ['data/lakehouse/normalized/rates.daily_rates/2026/04/part-00000.parquet'],
 'daily_rows': 2,
 'lpr_files': [],
 'lpr_rows': 0}

In [4]:
daily[[
    "field_name",
    "trade_date",
    "value",
    "unit",
    "field_role",
    "release_date",
    "effective_date",
    "revision_note",
    "source_caveat",
    "quality_status",
]] if not daily.empty else daily

,field_name,trade_date,value,unit,field_role,release_date,effective_date,revision_note,source_caveat,quality_status
0,shibor_1w,2026-04-20,1.328,percent,primary,2026-04-20,2026-04-20,low_revision_risk,tushare_wrapper_same_day_availability_caveat,pass_with_caveat
1,shibor_overnight,2026-04-20,1.221,percent,confirmatory,2026-04-20,2026-04-20,low_revision_risk,tushare_wrapper_same_day_availability_caveat,pass_with_caveat


In [5]:
raw_lpr_files = sorted((RAW_ROOT / "rates.lpr").glob("*/*/*.parquet"))
raw_lpr = (
    pd.concat([pd.read_parquet(path) for path in raw_lpr_files], ignore_index=True)
    if raw_lpr_files
    else pd.DataFrame()
)

{
    "raw_lpr_files": [str(path.relative_to(PROJECT_ROOT)) for path in raw_lpr_files],
    "raw_lpr_rows": len(raw_lpr),
    "normalized_lpr_files": [str(path.relative_to(PROJECT_ROOT)) for path in lpr_files],
    "normalized_lpr_rows": len(lpr),
    "raw_lpr_columns": list(raw_lpr.columns),
}

{'raw_lpr_files': ['data/lakehouse/raw/rates.lpr/2026/04/batch-267.parquet',
  'data/lakehouse/raw/rates.lpr/2026/04/batch-268.parquet'],
 'raw_lpr_rows': 0,
 'normalized_lpr_files': [],
 'normalized_lpr_rows': 0,
 'raw_lpr_columns': ['date', '1y', '5y']}

In [6]:
DAILY_REQUIRED_COLUMNS = {
    "field_name",
    "trade_date",
    "value",
    "unit",
    "field_role",
    "release_date",
    "effective_date",
    "source_name",
    "raw_batch_id",
    "ingested_at",
    "revision_note",
    "source_caveat",
    "quality_status",
}
LPR_REQUIRED_COLUMNS = {
    "field_name",
    "quote_date",
    "value",
    "unit",
    "field_role",
    "release_date",
    "effective_date",
    "source_name",
    "raw_batch_id",
    "ingested_at",
    "revision_note",
    "source_caveat",
    "quality_status",
}


def non_empty_text(frame: pd.DataFrame, column: str) -> bool:
    return column in frame.columns and frame[column].notna().all() and frame[column].astype(str).str.strip().ne("").all()


daily_expected_roles = {"shibor_1w": "primary", "shibor_overnight": "confirmatory"}
daily_role_ok = (
    not daily.empty
    and daily.set_index("field_name")["field_role"].to_dict() == daily_expected_roles
)
daily_release_ok = (
    not daily.empty
    and pd.to_datetime(daily["release_date"]).dt.date.eq(pd.to_datetime(daily["trade_date"]).dt.date).all()
)
daily_effective_ok = (
    not daily.empty
    and pd.to_datetime(daily["effective_date"]).dt.date.eq(pd.to_datetime(daily["trade_date"]).dt.date).all()
)

stage_f_checks = pd.DataFrame(
    [
        {
            "requirement": "rates.daily_rates has normalized canonical rows",
            "status": "pass" if len(daily) > 0 else "fail",
            "evidence": f"{len(daily)} rows across {len(daily_files)} normalized partition files",
        },
        {
            "requirement": "daily rates required columns are present",
            "status": "pass" if DAILY_REQUIRED_COLUMNS.issubset(daily.columns) else "fail",
            "evidence": str(sorted(DAILY_REQUIRED_COLUMNS - set(daily.columns))),
        },
        {
            "requirement": "daily rates latest-known business key is unique",
            "status": "pass" if not daily.duplicated(["field_name", "trade_date"]).any() else "fail",
            "evidence": f"duplicate keys: {int(daily.duplicated(['field_name', 'trade_date']).sum())}",
        },
        {
            "requirement": "daily rates field roles match v1 role map",
            "status": "pass" if daily_role_ok else "fail",
            "evidence": str(daily.set_index("field_name")["field_role"].to_dict()) if not daily.empty else "no rows",
        },
        {
            "requirement": "daily rates release/effective date timing is explicit",
            "status": "pass" if daily_release_ok and daily_effective_ok else "fail",
            "evidence": "release_date and effective_date equal quote trade_date for available Shibor rows",
        },
        {
            "requirement": "daily rates source caveat and revision note are visible",
            "status": "pass" if non_empty_text(daily, "source_caveat") and non_empty_text(daily, "revision_note") else "fail",
            "evidence": str(daily[["revision_note", "source_caveat", "quality_status"]].drop_duplicates().to_dict("records")) if not daily.empty else "no rows",
        },
        {
            "requirement": "rates.lpr has lpr_1y/lpr_5y canonical rows for the inspected window",
            "status": "gap" if lpr.empty else "pass",
            "evidence": f"{len(raw_lpr)} raw rows and {len(lpr)} normalized rows; source returned an empty LPR payload for this local run",
        },
        {
            "requirement": "LPR required columns can be assessed",
            "status": "not_applicable" if lpr.empty else ("pass" if LPR_REQUIRED_COLUMNS.issubset(lpr.columns) else "fail"),
            "evidence": "no normalized LPR rows to validate" if lpr.empty else str(sorted(LPR_REQUIRED_COLUMNS - set(lpr.columns))),
        },
    ]
)

stage_f_checks

,requirement,status,evidence
0,rates.daily_rates has normalized canonical rows,pass,2 rows across 1 normalized partition files
1,daily rates required columns are present,pass,[]
2,daily rates latest-known business key is unique,pass,duplicate keys: 0
3,daily rates field roles match v1 role map,pass,"{'shibor_1w': 'primary', 'shibor_overnight': '..."
4,daily rates release/effective date timing is e...,pass,release_date and effective_date equal quote tr...
5,daily rates source caveat and revision note ar...,pass,"[{'revision_note': 'low_revision_risk', 'sourc..."
6,rates.lpr has lpr_1y/lpr_5y canonical rows for...,gap,0 raw rows and 0 normalized rows; source retur...
7,LPR required columns can be assessed,not_applicable,no normalized LPR rows to validate
